### Setup

In [1]:
%load_ext autoreload
%autoreload 2

# Imports
import numpy as np, sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

import art.attacks.evasion.projected_gradient_descent.projected_gradient_descent_pytorch as _pgd_pt
_pgd_pt.compute_success = lambda *a, **kw: 0.0

sys.path.append(str(Path.cwd().parents[1]))

from utils.functions import get_windowed_data, denormalize
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

In [2]:
## Inputs
checkpoint_name, data_name, save_name, _ = FilenameLoader.const_pos()

checkpoint_file= f"../../saved_models/{checkpoint_name}"
data_file = f"../../data/{data_name}"
save_path = f"../final_data/{save_name}"

In [3]:
## Load the data that will be perturbed
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

### Checks

In [4]:
## Check that scaler works and matches og data
# Create the original, not normalized data
(x_train_og, y_train_og), (x_test_og, y_test_og), _, _ = get_windowed_data(data_file, 
                                                                      normalize=False, # NOTE: this is false 
                                                                      train_perc=80)

# Diff should be close to 0
maxes = []
for idx in range(len(x_test_og)):
    out1 = x_test_og[idx]
    out2 = denormalize(x_test[idx], scaler)

    out3 = out2 - out1.numpy()
    out4 = out3.flatten()
    maxes.append(max(out4))
max(maxes)

np.float64(0.0013367487808864098)

### Attack Generation

#### Setup
Define the is_window_constant function

In [5]:
def is_window_constant(x_test, x_test_adv, index, scaler, threshold=10):
    """
    Inputs: 
    - x_test - the original, clean x_test 
    - x_test_adv - the adversarially perturbed data
    - index - which window in the x_test's that're be compared
        (the window is not directly passed in because indicies are easier to handle
        when testing)
    - scaler - scaler used to scale the original data
    - threshold - the farther two points can be away from each other. 

    Output:
    - [bool, float] - tuple of whether the window passes the constant check, and
        what the max distance was

    The function denormalizes both x_test and y_test, takes their difference, 
    and calculates whether the biggest and smallest change are within the threshold.
    """
    # Denormalized (original units) versions of the same values
    x_denorm = denormalize(x_test[index], scaler)
    x_adv_denorm = denormalize(x_test_adv[index], scaler)

    diff = x_denorm - x_adv_denorm
    points = diff[:, 1:3]  # (10, 2) -> (dx, dy) per message

    max_pairwise = max(
        np.linalg.norm(points[i] - points[j])
        for i in range(len(points)) for j in range(i + 1, len(points))
    )

    return max_pairwise < threshold, max_pairwise


Define model (to run later x_test_adv) and classifier (an input to the adversarial attack).

In [6]:
model, classifier = get_model_classifier(checkpoint_file)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


#### Run analysis

Create x_test_adv

In [7]:
eps = 0.0012
attack = FastGradientMethod(estimator=classifier, eps=eps, targeted=True)
x_test_adv = attack.generate(x=x_test.numpy(), y=1 - y_test.numpy())
# the 1-y is so the targeted=True pushes the towards the opposite label

Find the indices that are labeled as attackers, use algorithm

In [ ]:
# Define threshold
threshold = 10
print("Current eps:", eps, "\nCurrent threshold:", threshold)

# Run is_constant on every attacker window
attack_indices = []
tr, fa = 0, 0 # true constants, and false constants
diffs = []
for i in range(len(y_test)):
    if (y_test[i] == 1).all():
        attack_indices.append(i)
        is_const, max_diff = is_window_constant(x_test, x_test_adv, i, scaler, threshold=threshold)
        if(is_const): tr += 1
        else: fa += 1
        diffs.append(max_diff)

## Print results
lst, freqs = np.unique(diffs, return_counts=True)
print("unique diffs", lst)
print("unique freqs", freqs)
print("max", round(max(diffs), 4), "m")
print("tr:", tr, "| fa:", fa)
print("% of non-const windows:", str(round(fa/(fa+tr)*100, 4)) + "%")


Current eps: 0.0012
Current threshold: 10
unique diffs [0.00000000e+00 1.77635684e-15 5.32907052e-15 ... 4.94359409e+00
 4.94359409e+00 4.94359409e+00]
unique freqs [1797    1    1 ...    5    2    2]
max 4.9436 m
tr: 37070 | fa: 0
% of non-const windows: 0.0%


#### Run x_test_adv on model

To ensure that the attack affects the model performance

In [8]:
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test_adv, y_test=y_test.numpy(), # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=None, filename=None, # saving information
                save_results=False,
                collapsed=False
                ) 

# Check if output fails (it should fail)
print('f1', out['wrapper']['f1'])
print('fpr', out['wrapper']['falsePositiveRate'])
print('fnr', out['wrapper']['falseNegativeRate'])

AttributeError: 'numpy.ndarray' object has no attribute 'numpy'